###=========================================================
### UI แปลงสี (RGB → HEX/HSL/CMYK/LCH) + สวอตช์ บน Google Colab
### - ไม่พึ่ง lib ภายนอก (pure-Python + Pillow + ipywidgets)
### =========================================================


In [ ]:
!pip -q install pillow ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 18.8 MB/s eta 0:00:00


In [ ]:
import re, io, math
import pandas as pd
from math import sqrt, pow, atan2, degrees, sin, cos, exp
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display, HTML
import ipywidgets as w

# ----------------------------
# Parsing utilities
# ----------------------------
def parse_color_any(s):
    """
    Accept strings:
      - "#RRGGBB" or "#RGB"
      - "rgb(r,g,b)"
      - "hsl(h, s%, l%)"
      - "cmyk(c%, m%, y%, k%)"
      - "lch(L, C, H)"  # L in 0..100, C>=0, H in deg
    Return (r,g,b) in 0..255
    """
    s = s.strip()

    # HEX
    if s.startswith("#"):
        return hex_to_rgb(s)

    # rgb(...)
    m = re.match(r"rgb\s*\(\s*([0-9]+)\s*,\s*([0-9]+)\s*,\s*([0-9]+)\s*\)", s, re.I)
    if m:
        r, g, b = map(int, m.groups())
        return clamp_rgb((r, g, b))

    # hsl(...)
    m = re.match(r"hsl\s*\(\s*([0-9.]+)\s*,\s*([0-9.]+)\s*%\s*,\s*([0-9.]+)\s*%\s*\)", s, re.I)
    if m:
        h, ss, ll = map(float, m.groups())
        return hsl_to_rgb(h % 360, max(0,min(100,ss)), max(0,min(100,ll)))

    # cmyk(...)
    m = re.match(r"cmyk\s*\(\s*([0-9.]+)\s*%\s*,\s*([0-9.]+)\s*%\s*,\s*([0-9.]+)\s*%\s*,\s*([0-9.]+)\s*%\s*\)", s, re.I)
    if m:
        c, m_, y, k = map(float, m.groups())
        return cmyk_to_rgb(c, m_, y, k)

    # lch(...)
    m = re.match(r"lch\s*\(\s*([0-9.]+)\s*,\s*([0-9.]+)\s*,\s*([\-0-9.]+)\s*\)", s, re.I)
    if m:
        L, C, H = map(float, m.groups())
        return lch_to_rgb((L, C, H % 360))

    raise ValueError("รูปแบบสีไม่รองรับ: ใช้ #RRGGBB / rgb() / hsl() / cmyk() / lch()")

def clamp_rgb(rgb):
    r, g, b = rgb
    return (max(0,min(255,int(round(r)))), max(0,min(255,int(round(g)))), max(0,min(255,int(round(b)))))

# ----------------------------
# Conversions
# ----------------------------
def rgb_to_hex(r, g, b):
    return "#{:02X}{:02X}{:02X}".format(*clamp_rgb((r,g,b)))

def hex_to_rgb(hex_str):
    s = hex_str.strip().lstrip('#')
    if len(s) == 3:
        s = ''.join(ch*2 for ch in s)
    if len(s) != 6:
        raise ValueError("HEX ต้องเป็น 3 หรือ 6 หลัก")
    return (int(s[0:2],16), int(s[2:4],16), int(s[4:6],16))

def rgb_to_hsl(r, g, b):
    r1, g1, b1 = r/255.0, g/255.0, b/255.0
    cmax, cmin = max(r1, g1, b1), min(r1, g1, b1)
    delta = cmax - cmin
    L = (cmax + cmin)/2.0
    if delta == 0:
        H = 0.0; S = 0.0
    else:
        S = delta / (1 - abs(2*L - 1))
        if cmax == r1:
            H = ((g1 - b1)/delta) % 6
        elif cmax == g1:
            H = ((b1 - r1)/delta) + 2
        else:
            H = ((r1 - g1)/delta) + 4
        H *= 60.0
    return (H % 360, S*100, L*100)

def hsl_to_rgb(h, s, l):
    s /= 100.0; l /= 100.0
    C = (1 - abs(2*l - 1)) * s
    X = C * (1 - abs((h/60.0) % 2 - 1))
    m = l - C/2.0
    if   0 <= h < 60:   rp,gp,bp = C, X, 0
    elif 60 <= h < 120: rp,gp,bp = X, C, 0
    elif 120 <= h < 180:rp,gp,bp = 0, C, X
    elif 180 <= h < 240:rp,gp,bp = 0, X, C
    elif 240 <= h < 300:rp,gp,bp = X, 0, C
    else:               rp,gp,bp = C, 0, X
    r = (rp + m) * 255; g = (gp + m) * 255; b = (bp + m) * 255
    return clamp_rgb((r,g,b))

def rgb_to_cmyk(r, g, b):
    r1, g1, b1 = r/255.0, g/255.0, b/255.0
    c = 1 - r1; m = 1 - g1; y = 1 - b1
    k = min(c, m, y)
    if k >= 1.0 - 1e-12:
        return (0.0, 0.0, 0.0, 100.0)
    c = (c - k) / (1 - k)
    m = (m - k) / (1 - k)
    y = (y - k) / (1 - k)
    return (c*100, m*100, y*100, k*100)

def cmyk_to_rgb(c, m, y, k):
    # c,m,y,k in %
    C, M, Y, K = c/100.0, m/100.0, y/100.0, k/100.0
    r = 255 * (1 - C) * (1 - K)
    g = 255 * (1 - M) * (1 - K)
    b = 255 * (1 - Y) * (1 - K)
    return clamp_rgb((r,g,b))

# sRGB gamma helpers
def _srgb_to_linear_01(c01):
    if c01 <= 0.04045:
        return c01 / 12.92
    return ((c01 + 0.055) / 1.055) ** 2.4

def _linear01_to_srgb01(c):
    if c <= 0.0031308:
        return 12.92 * c
    return 1.055 * (c ** (1/2.4)) - 0.055

def rgb_to_xyz(r, g, b):
    r01, g01, b01 = r/255.0, g/255.0, b/255.0
    r_lin = _srgb_to_linear_01(r01)
    g_lin = _srgb_to_linear_01(g01)
    b_lin = _srgb_to_linear_01(b01)
    X = 0.4124564*r_lin + 0.3575761*g_lin + 0.1804375*b_lin
    Y = 0.2126729*r_lin + 0.7151522*g_lin + 0.0721750*b_lin
    Z = 0.0193339*r_lin + 0.1191920*g_lin + 0.9503041*b_lin
    return (X, Y, Z)

def xyz_to_rgb(X, Y, Z):
    # inverse matrix
    r_lin =  3.2404542*X - 1.5371385*Y - 0.4985314*Z
    g_lin = -0.9692660*X + 1.8760108*Y + 0.0415560*Z
    b_lin =  0.0556434*X - 0.2040259*Y + 1.0572252*Z
    r01 = _linear01_to_srgb01(r_lin)
    g01 = _linear01_to_srgb01(g_lin)
    b01 = _linear01_to_srgb01(b_lin)
    return clamp_rgb((r01*255, g01*255, b01*255))

def xyz_to_lab(X, Y, Z):
    Xn, Yn, Zn = 0.95047, 1.0, 1.08883
    x, y, z = X/Xn, Y/Yn, Z/Zn
    eps = 216/24389
    kappa = 24389/27
    def f(t): return t**(1/3) if t > eps else (kappa*t + 16)/116
    fx, fy, fz = f(x), f(y), f(z)
    L = 116*fy - 16
    a = 500*(fx - fy)
    b = 200*(fy - fz)
    return (L, a, b)

def lab_to_xyz(L, a, b):
    Xn, Yn, Zn = 0.95047, 1.0, 1.08883
    fy = (L + 16) / 116
    fx = fy + (a / 500)
    fz = fy - (b / 200)
    def finv(t):
        if t**3 > 216/24389:
            return t**3
        return (116*t - 16) / (24389/27)
    x = finv(fx); y = finv(fy); z = finv(fz)
    return (x*Xn, y*Yn, z*Zn)

def lab_to_lch(L, a, b):
    C = sqrt(a*a + b*b)
    H = (degrees(atan2(b, a))) % 360.0
    return (L, C, H)

def lch_to_lab(L, C, H):
    hr = math.radians(H)
    a = C * math.cos(hr)
    b = C * math.sin(hr)
    return (L, a, b)

def rgb_to_lch(r, g, b):
    X, Y, Z = rgb_to_xyz(r, g, b)
    L, a, bb = xyz_to_lab(X, Y, Z)
    C = sqrt(a*a + bb*bb)
    H = (degrees(atan2(bb, a))) % 360.0
    return (L, C, H)

def lch_to_rgb(lch):
    L, C, H = lch
    a = C * math.cos(math.radians(H))
    b = C * math.sin(math.radians(H))
    X, Y, Z = lab_to_xyz(L, a, b)
    return xyz_to_rgb(X, Y, Z)

# ----------------------------
# Color Differences
# ----------------------------
def deltaE76_rgb(rgb1, rgb2):
    L1,a1,b1 = xyz_to_lab(*rgb_to_xyz(*rgb1))
    L2,a2,b2 = xyz_to_lab(*rgb_to_xyz(*rgb2))
    return sqrt((L1-L2)**2 + (a1-a2)**2 + (b1-b2)**2)

def deltaE2000_rgb(rgb1, rgb2):
    # Implementation adapted from Sharma et al. 2005
    L1, a1, b1 = xyz_to_lab(*rgb_to_xyz(*rgb1))
    L2, a2, b2 = xyz_to_lab(*rgb_to_xyz(*rgb2))

    avg_L = (L1 + L2) / 2.0
    C1 = sqrt(a1*a1 + b1*b1)
    C2 = sqrt(a2*a2 + b2*b2)
    avg_C = (C1 + C2) / 2.0

    G = 0.5 * (1 - sqrt((avg_C**7) / (avg_C**7 + 25**7)))
    a1p = (1+G) * a1
    a2p = (1+G) * a2
    C1p = sqrt(a1p*a1p + b1*b1)
    C2p = sqrt(a2p*a2p + b2*b2)
    avg_Cp = (C1p + C2p)/2.0

    def hp_fun(a, b):
        if a == 0 and b == 0:
            return 0.0
        hp = degrees(atan2(b, a)) % 360.0
        return hp

    h1p = hp_fun(a1p, b1)
    h2p = hp_fun(a2p, b2)

    dLp = L2 - L1
    dCp = C2p - C1p

    if C1p*C2p == 0:
        dhp = 0.0
    else:
        dhp = h2p - h1p
        if dhp > 180:
            dhp -= 360
        elif dhp < -180:
            dhp += 360
    dHp = 2 * sqrt(C1p*C2p) * sin(math.radians(dhp/2.0))

    avg_Lp = (L1 + L2) / 2.0
    if C1p*C2p == 0:
        avg_hp = h1p + h2p
    else:
        if abs(h1p - h2p) > 180:
            avg_hp = (h1p + h2p + 360) / 2.0 if (h1p + h2p) < 360 else (h1p + h2p - 360)/2.0
        else:
            avg_hp = (h1p + h2p) / 2.0

    T = 1 - 0.17*cos(math.radians(avg_hp - 30)) + 0.24*cos(math.radians(2*avg_hp)) + \
        0.32*cos(math.radians(3*avg_hp + 6)) - 0.20*cos(math.radians(4*avg_hp - 63))

    d_ro = 30 * exp(-(((avg_hp - 275)/25)**2))
    Rc = 2 * sqrt((avg_Cp**7) / (avg_Cp**7 + 25**7))
    Sl = 1 + (0.015 * ((avg_Lp - 50)**2)) / sqrt(20 + ((avg_Lp - 50)**2))
    Sc = 1 + 0.045 * avg_Cp
    Sh = 1 + 0.015 * avg_Cp * T
    Rt = -sin(math.radians(2 * d_ro)) * Rc

    return sqrt(
        (dLp / Sl)**2 +
        (dCp / Sc)**2 +
        (dHp / Sh)**2 +
        Rt * (dCp / Sc) * (dHp / Sh)
    )

def similarity_percent(rgb1, rgb2, method="DE2000", cap=50.0):
    """
    Map ΔE -> 0..100%:
      sim% = max(0, 100 * (1 - ΔE / cap))
    cap: scaling (ΔE where similarity becomes 0). 50 is a practical default.
    """
    dE = deltaE2000_rgb(rgb1, rgb2) if method == "DE2000" else deltaE76_rgb(rgb1, rgb2)
    return max(0.0, 100.0 * (1.0 - (dE / cap))), dE

# ----------------------------
# Swatch image (base + tints/shades)
# ----------------------------
def tweak_lightness_rgb(r, g, b, delta_l):
    H,S,L = rgb_to_hsl(r,g,b)
    L2 = max(0,min(100,L + delta_l))
    return hsl_to_rgb(H, S, L2)

def make_palette_image(name, r, g, b):
    swatches = [
        ("Shade 40", tweak_lightness_rgb(r,g,b, -40)),
        ("Shade 20", tweak_lightness_rgb(r,g,b, -20)),
        ("Base",     (r,g,b)),
        ("Tint 20",  tweak_lightness_rgb(r,g,b, +20)),
        ("Tint 40",  tweak_lightness_rgb(r,g,b, +40)),
    ]
    width_per = 220; height = 160; margin = 8
    img = Image.new("RGB", (width_per * len(swatches), height), "white")
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype("DejaVuSans.ttf", 16)
        font_small = ImageFont.truetype("DejaVuSans.ttf", 14)
    except:
        font = ImageFont.load_default(); font_small = ImageFont.load_default()

    for i,(label,(rr,gg,bb)) in enumerate(swatches):
        x0 = i * width_per
        draw.rectangle((x0+margin, margin, x0+width_per-margin, height-48), fill=(rr,gg,bb))
        hexc = rgb_to_hex(rr,gg,bb)
        ytxt = height - 44
        draw.text((x0+margin, ytxt), f"{name} • {label}", fill="black", font=font)
        draw.text((x0+margin, ytxt+18), hexc, fill="black", font=font_small)
    bio = io.BytesIO(); img.save(bio, format="PNG")
    return bio.getvalue()

# ----------------------------
# UI Widgets
# ----------------------------
section_title = lambda t: w.HTML(f"<h2 style='margin:60px 0 30px'>{t}</h2>")

# Base color input (any format)
base_in = w.Text(value="#3B82F6", description="Base color")
token_in = w.Text(value="Primary", description="Name")
btn_apply_base = w.Button(description="Apply & Convert", button_style="primary", icon="refresh")

# Two-color compare
c1 = w.Text(value="#3B82F6", description="Color A")
c2 = w.Text(value="rgb(59,130,246)", description="Color B")
metric = w.Dropdown(options=[("CIEDE2000","DE2000"), ("ΔE76","DE76")], value="DE2000", description="Metric")
cap = w.FloatSlider(value=50.0, min=10.0, max=100.0, step=1.0, description="ΔE cap")
out_two = w.Output()

# Multi list compare
list_colors = w.Textarea(
    value="#22C55E\nhsl(38, 92%, 50%)\nrgb(239,68,68)\ncmyk(64%, 43%, 0%, 19%)\nlch(70,40,220)",
    description="Candidates",
    layout=w.Layout(width="100%", height="130px")
)
min_sim = w.IntSlider(value=70, min=0, max=100, step=1, description="Min % Similarity")
btn_run_list = w.Button(description="Compare List", button_style="success", icon="search")
out_base = w.Output()
out_list = w.Output()

# ----------------------------
# Helpers for colored Similarity badges
# ----------------------------
def _sim_color(sim):
    if sim is None:
        return "inherit"
    try:
        s = float(sim)
    except:
        return "inherit"
    if s > 80:
        return "green"
    elif s >= 70:
        return "orange"
    else:
        return "red"

def _sim_span(sim):
    color = _sim_color(sim)
    try:
        s = float(sim)
        txt = f"{s:.2f}%"
    except:
        txt = str(sim)
    return f"<span style='color:{color};font-weight:600'>{txt}</span>"

# ----------------------------
# Convert report card
# ----------------------------
def convert_report(rgb, name):
    r,g,b = rgb
    H,S,L = rgb_to_hsl(r,g,b)
    C,M,Y,K = rgb_to_cmyk(r,g,b)
    Lc,Cc,Hc = rgb_to_lch(r,g,b)
    hx = rgb_to_hex(r,g,b)

    df = pd.DataFrame([{
        "Name": name,
        "RGB": f"rgb({r},{g},{b})",
        "HEX": hx,
        "HSL": f"hsl({H:.2f}, {S:.2f}%, {L:.2f}%)",
        "CMYK": f"({C:.2f}%, {M:.2f}%, {Y:.2f}%, {K:.2f}%)",
        "LCH": f"({Lc:.4f}, {Cc:.4f}, {Hc:.4f})"
    }])
    card = HTML(f"""
      <div style="display:flex;align-items:center;gap:12px;margin:8px 0;">
        <div style="width:52px;height:52px;border-radius:8px;border:1px solid #ddd;background:{hx};"></div>
        <div>
          <div style="font-weight:600">{name}</div>
          <div style="font-size:12px;color:#666">{hx} • rgb({r},{g},{b}) • hsl({H:.2f}, {S:.2f}%, {L:.2f}%)</div>
        </div>
      </div>
    """)
    return card, df

# ----------------------------
# Event handlers
# ----------------------------
def on_apply_base(_=None):
    try:
        rgb = parse_color_any(base_in.value)
    except Exception as e:
        with out_base: out_base.clear_output(); print("⚠️", e); return

    with out_base:
        out_base.clear_output()
        card, df = convert_report(rgb, token_in.value or "Base")
        display(card); display(df)

        # swatches
        img_bytes = make_palette_image(token_in.value or "Base", *rgb)
        display(Image.open(io.BytesIO(img_bytes)))

def on_compare_two(_=None):
    try:
        rgb1 = parse_color_any(c1.value)
        rgb2 = parse_color_any(c2.value)
    except Exception as e:
        with out_two: out_two.clear_output(); print("⚠️", e); return

    sim, dE = similarity_percent(rgb1, rgb2, method=metric.value, cap=cap.value)
    with out_two:
        out_two.clear_output()
        sim_badge = _sim_span(sim)
        display(HTML(
            f"<div style='margin:30px 0 60px; font-size:20px;'>"
            f"<b>Similarity:</b> {sim_badge} &nbsp; | &nbsp; "
            f"<b>ΔE ({metric.label}):</b> {dE:.3f} &nbsp; | &nbsp; "
            f"<b>cap:</b> {cap.value}"
            f"</div>"
        ))
        cardA, dfA = convert_report(rgb1, "Color A")
        cardB, dfB = convert_report(rgb2, "Color B")
        display(cardA); display(dfA)
        display(cardB); display(dfB)

def on_compare_list(_=None):
    try:
        base_rgb = parse_color_any(base_in.value)
    except Exception as e:
        with out_list: out_list.clear_output(); print("⚠️", e); return

    rows = []
    for line in list_colors.value.splitlines():
        s = line.strip()
        if not s: continue
        try:
            rgb = parse_color_any(s)
            sim, dE = similarity_percent(base_rgb, rgb, method=metric.value, cap=cap.value)
            rows.append({
                "Input": s,
                "HEX": rgb_to_hex(*rgb),
                "RGB": f"rgb{rgb}",
                "Similarity%": round(sim, 2),
                "ΔE": round(dE, 3)
            })
        except Exception as e:
            rows.append({"Input": s, "HEX": "-", "RGB": "-", "Similarity%": "-", "ΔE": f"Error: {e}"})

    # Filter & sort
    df = pd.DataFrame(rows)
    def safe_val(x):
        try: return float(x)
        except: return 9999
    df["__sort"] = df["ΔE"].apply(safe_val)
    if isinstance(min_sim.value, (int,float)):
        def pass_thr(x):
            try: return float(x) >= min_sim.value
            except: return False
        df = df[df["Similarity%"].apply(pass_thr)]
    df = df.sort_values("__sort").drop(columns="__sort")

    with out_list:
        out_list.clear_output()
        base_hex = rgb_to_hex(*base_rgb)
        display(HTML(
            f"<div><b>Base:</b> {base_hex} &nbsp; • &nbsp; {token_in.value or 'Base'} "
            f"&nbsp; • &nbsp; Metric: <code>{metric.label}</code> "
            f"&nbsp; • &nbsp; cap: {cap.value}</div>"
        ))

        # ทำให้คอลัมน์ Similarity% เป็น HTML มีสี
        df_html = df.copy()
        df_html["Similarity%"] = df_html["Similarity%"].apply(_sim_span)

        # แสดงตารางแบบไม่ escape HTML เพื่อให้สีแสดงผล
        display(HTML(df_html.to_html(escape=False, index=False)))

# Bind events
btn_apply_base.on_click(on_apply_base)
c1.observe(on_compare_two, "value")
c2.observe(on_compare_two, "value")
metric.observe(on_compare_two, "value")
cap.observe(on_compare_two, "value")
btn_run_list.on_click(on_compare_list)
metric.observe(lambda _: (on_apply_base(), on_compare_list()), "value")
cap.observe(lambda _: (on_apply_base(), on_compare_list()), "value")
min_sim.observe(lambda _: on_compare_list(), "value")

# ----------------------------
# Layout
# ----------------------------
ui = w.VBox([
    section_title("1) Base Color & Conversions"),
    w.HBox([base_in, token_in, btn_apply_base]),
    w.HTML("<p style='color:red;'>รับค่า:<ul><li>HEX: #RRGGBB</li><li>RGB: rgb(r,g,b)</li><li>HSL: hsl(h,s%,l%)</li><li>CMYK: cmyk(c%,m%,y%,k%)</li><li>LCH:  lch(L,C,H)</li></ul></p>"),
    w.HTML("<hr>"),

    section_title("2) Compare Two Colors"),
    w.HBox([c1, c2]),
    w.HBox([metric, cap]),
    out_two,
    w.HTML("<hr>"),

    # section_title("3) Compare Base vs Multiple Colors"),
    # list_colors,
    # w.HBox([min_sim, metric, cap, btn_run_list]),
    # out_base,
    # out_list
])

display(ui)
on_apply_base()
on_compare_two()
on_compare_list()
